# Persistence & Async Scaling: Проектирование устойчивых агентов

Этот блокнот демонстрирует работу асинхронных циклов и механизмов сохранения состояния (Checkpointing).

### 1. Установка зависимостей
Нам понадобятся `langgraph` для графов и асинхронные библиотеки для имитации работы высоконагруженной системы.

In [1]:
!pip install -U langgraph langchain pydantic langgraph-checkpoint-sqlite aiosqlite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.6/151.6 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
  

### 2. Определение состояния (State) и защита от Race Conditions
В параллельных системах (например, когда 5 роботов на заводе одновременно докладывают о состоянии), важно, чтобы данные не перезаписывались. Мы используем `operator.add` для списков, чтобы LangGraph автоматически объединял результаты.

**Объяснение:** В продакшене использование простых строк (`str`) для параллельных узлов приведет к потере данных. Мы используем списки, чтобы сохранить каждый "голос" системы.

In [2]:
import operator
from typing import Annotated, List, Union
from typing_extensions import TypedDict

class IndustrialState(TypedDict):
    # Сообщения будут добавляться в список, а не перезаписываться
    messages: Annotated[list, operator.add]
    # Список найденных дефектов (для примера с фабрикой)
    defects: Annotated[List[str], operator.add]
    # Флаг подтверждения инженером
    engineer_approval: bool

print("✅ State спроектирован для параллельной работы.")

✅ State спроектирован для параллельной работы.


### 3. Асинхронные узлы (Async Nodes)
В 2026 году ожидание ответа от LLM (2-5 сек) — это I/O операция. Чтобы не вешать сервер, все узлы должны быть `async`.

In [3]:
import asyncio
import random

async def sensor_scanner(state: IndustrialState):
    """Имитация работы сканера на конвейере автомобильного завода."""
    sensor_id = random.randint(1, 100)
    print(f"📡 Датчик {sensor_id}: Начало сканирования...")
    await asyncio.sleep(2) # Имитируем долгий инференс модели Qwen3
    defect = f"Дефект # {random.randint(1000, 9999)} на линии {sensor_id}"
    return {"defects": [defect]}

async def aggregator_node(state: IndustrialState):
    """Сбор всех данных в единый отчет."""
    print(f"📊 Агрегатор: Получено дефектов: {len(state['defects'])}")
    summary = f"Всего обнаружено проблем: {len(state['defects'])}"
    return {"messages": [("ai", summary)]}

print("✅ Асинхронные узлы готовы.")

✅ Асинхронные узлы готовы.


### 4. Динамический Fan-Out (Send API)
Паттерн Map-Reduce. Представьте, что нам нужно проверить 3 разных параметра двигателя одновременно. Мы не пишем 3 узла, а динамически "размножаем" одну задачу.

**Объяснение:** Код ниже — это рабочий пример API `Send`. В продакшене это позволяет масштабироваться на сотни параллельных проверок.

In [4]:
from langgraph.types import Send

def parallel_router(state: IndustrialState):
    """Решает запустить 3 параллельных сканера."""
    # В реальной жизни количество задач может зависеть от сложности запроса
    return [Send("scanner", state) for _ in range(3)]

print("✅ Логика параллельного распределения (Fan-Out) создана.")

✅ Логика параллельного распределения (Fan-Out) создана.


### 5. Persistence: Настройка хранилища (SQLite)
Чтобы агент "помнил" нас после перезагрузки, мы подключаем чекпоинтер. В Colab мы используем файл `checkpoints.db`.

**Объяснение:** Это **функциональный эквивалент** AsyncPostgresSaver из слайдов. Если процесс упадет, данные сохранятся в этом файле.

In [7]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# 1. Создаем контекстный менеджер
connection_context = AsyncSqliteSaver.from_conn_string("checkpoints.db")

# 2. Входим в него через await, чтобы получить сам объект BaseCheckpointSaver
# В обычном коде это делается через 'async with', но в ноутбуке нам нужен глобальный объект
checkpointer = await connection_context.__aenter__()

print("✅ Асинхронный чекпоинтер успешно извлечен и готов к работе.")

✅ Асинхронный чекпоинтер успешно извлечен и готов к работе.


### 6. Сборка графа с Human-in-the-Loop
Мы собираем граф, который делает параллельный поиск, а затем **замирает** перед финальным отчетом, ожидая аппрува инженера (HITL).

In [8]:
from langgraph.graph import StateGraph, END

builder = StateGraph(IndustrialState)

builder.add_node("scanner", sensor_scanner)
builder.add_node("aggregator", aggregator_node)

# Вход в параллель
builder.set_conditional_entry_point(
    parallel_router,
    {"scanner": "scanner"}
)

# Выход из параллели в агрегатор
builder.add_edge("scanner", "aggregator")

# Компиляция с ПРЕРЫВАНИЕМ перед агрегатором
# Это имитирует паттерн "Безопасность прежде всего"
app = builder.compile(
    checkpointer=checkpointer, # Теперь здесь правильный тип данных
    interrupt_before=["aggregator"]
)

print("✅ Граф скомпилирован.")

✅ Граф скомпилирован.


### 7. Тестирование: Запуск и Резюме
Запустим систему. Она должна выполнить сканирование и остановиться.

**Объяснение:** Мы передаем `thread_id`. Это ключ к контексту. Если вы запустите эту ячейку снова с тем же ID, система поймет, что она на паузе.

In [9]:
config = {"configurable": {"thread_id": "factory-line-v1"}}

async def run_test():
    # Начальный запуск
    print("🚀 Запуск процесса...")
    async for event in app.astream({"messages": [("human", "Проверь линию")]}, config):
        print(event)

await run_test()

print("\n⚠️ СИСТЕМА ОСТАНОВЛЕНА: Ожидание подтверждения инженера (HITL).")

🚀 Запуск процесса...
📡 Датчик 85: Начало сканирования...
📡 Датчик 7: Начало сканирования...
📡 Датчик 39: Начало сканирования...
{'scanner': {'defects': ['Дефект # 4311 на линии 85']}}
{'scanner': {'defects': ['Дефект # 2525 на линии 7']}}
{'scanner': {'defects': ['Дефект # 8071 на линии 39']}}
{'__interrupt__': ()}

⚠️ СИСТЕМА ОСТАНОВЛЕНА: Ожидание подтверждения инженера (HITL).


### 8. Time Travel: Просмотр истории
Мы можем посмотреть, что "думал" агент на каждом шаге, прежде чем он встал на паузу.

In [10]:
state_history = []
async for state in app.aget_state_history(config):
    state_history.append(state)

print(f"Количество шагов в истории: {len(state_history)}")
print(f"Последние найденные дефекты: {state_history[0].values['defects']}")

Количество шагов в истории: 3
Последние найденные дефекты: ['Дефект # 4311 на линии 85', 'Дефект # 2525 на линии 7', 'Дефект # 8071 на линии 39']


### Домашнее задание: News Digest Agent

**Контекст:** Вы создаете систему мониторинга для юридического департамента, которая собирает новости о новых законах.

**Задача:**
1.  **Модифицируйте State:** Добавьте поле `urls: List[str]`.
2.  **Реализуйте Map-Phase:** Создайте узел, который принимает 1 тему и возвращает список из 3 URL (заглушек).
3.  **Параллельная обработка:** Используйте `Send API`, чтобы параллельно "обработать" каждый URL (каждый узел должен просто вернуть название статьи и `await asyncio.sleep(1)`).
4.  **HITL:** Настройте граф так, чтобы он прерывался **после** того, как все статьи собраны, но **до** того, как будет сформирован финальный PDF-отчет.
5.  **Time Travel:** Напишите код, который выводит список всех заголовков статей из истории чекпоинтов.

**Что прислать:** Ссылку на ваш Colab или скриншот вывода, где видны параллельные логи "Сканирование URL..." и статус прерывания.

---
*Архитектурный совет: Всегда используйте `thread_id` для разделения задач разных юристов/отделов, чтобы избежать утечки данных между сессиями.*